# 08. TempoTracker: оценка темпа исполнителя

`TempoTracker` живёт в `output_dispatcher.py`. Это **не** часть Core ML и не часть HMM — он сидит **между** трекером и рендерером. Берёт на вход поток `(score_index, timestamp)` и выдаёт скаляр `tempo_ratio`:

- `tempo_ratio = 1.0` — солист играет в номинальном темпе;
- `tempo_ratio = 1.2` — солист на 20% быстрее (оркестр должен ускориться);
- `tempo_ratio = 0.8` — солист медленнее на 20% (оркестр замедлиться).

## Алгоритм

1. Хранит deque последних `history_size` контрольных точек `(score_position, time)`.
2. На каждом новом score-индексе считает `raw_ratio = nominal_elapsed / actual_elapsed` для каждой пары точек.
3. Берёт **медиану** по истории — это устойчиво к выбросам.
4. Применяет **deadzone** (не меняет ratio при малых отклонениях) и **EMA-сглаживание**.
5. Делает **idle reset** если давно не было новых событий.

Это ровно тот скаляр, который RL-агент тезисов **заменяет** (или дополняет) на предсказании `a_t ∈ [0.5, 2.0]`.

In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import matplotlib.pyplot as plt

from musictech.core import HybridScoreFollower, TempoTracker
from dataset_viewer import load_performance

plt.rcParams["figure.figsize"] = (10, 4)
plt.rcParams["axes.grid"] = True
DATA = PROJECT_ROOT / "generated_dataset"

In [ ]:
def run_with_tempo(score_path: Path, perf_path: Path):
    follower = HybridScoreFollower(str(score_path), load_tuning_profile=False)
    tempo = TempoTracker(str(score_path))
    performance = load_performance(perf_path)
    log = []
    for event in performance:
        idx = follower.process_event(event["pitch"], event["timestamp"])
        ratio = tempo.update(idx, event["timestamp"])
        log.append({
            "event_time": event["timestamp"],
            "score_index": idx,
            "tempo_ratio": ratio,
            "tempo_variance": tempo.last_variance,
        })
    return follower, tempo, log

## 1. На `ideal` темп должен быть около 1.0

In [ ]:
follower, tempo, log = run_with_tempo(DATA / "ideal.json", DATA / "ideal.mid")
times = [r["event_time"] for r in log]
ratios = [r["tempo_ratio"] for r in log]

fig, ax = plt.subplots()
ax.plot(times, ratios, marker="o")
ax.axhline(1.0, color="red", linestyle=":", label="nominal")
ax.set_xlabel("time, s")
ax.set_ylabel("tempo_ratio")
ax.set_title("TempoTracker on 'ideal' — should stay near 1.0")
ax.legend()
plt.show()

print(f"mean tempo_ratio: {np.mean(ratios):.3f}")
print(f"std tempo_ratio : {np.std(ratios):.3f}")

## 2. На `rubato` темп должен «дышать»

Помним: в `rubato.mid` пианист сначала ускоряется (короткие интервалы), потом замедляется. Ожидаем `tempo_ratio > 1.0` в середине, `< 1.0` в конце.

In [ ]:
follower, tempo, log = run_with_tempo(DATA / "rubato.json", DATA / "rubato.mid")
times = [r["event_time"] for r in log]
ratios = [r["tempo_ratio"] for r in log]

fig, ax = plt.subplots()
ax.plot(times, ratios, marker="o")
ax.axhline(1.0, color="red", linestyle=":", label="nominal")
ax.set_xlabel("time, s")
ax.set_ylabel("tempo_ratio")
ax.set_title("TempoTracker on 'rubato' — should breathe around 1.0")
ax.legend()
plt.show()

print(f"min/max tempo_ratio: {min(ratios):.3f} / {max(ratios):.3f}")

Из-за `min_nominal_window=0.18` и `history_size=5` на коротких пьесах темп успевает обновиться 2–3 раза. На реальных пьесах (4000+ нот) кривая темпа получится гораздо более плавной — это и есть прицельная зона для RL-агента, чтобы делать **опережающее** предсказание на 200–500 мс вперёд, как требуют тезисы.

## 3. Что в `TempoTracker` интересно для RL

Атрибуты, которые **уже** есть и которые войдут в `RLObservation`:

| В тезисах (`s_t`) | В `TempoTracker` |
|---|---|
| `τ_{t-K:t}` — K последних оценок темпа | `tempo.recent_tempo_ratios` (deque) |
| Variance | `tempo.last_variance` |
| История контрольных точек | `tempo.recent_control_points` |

Эмиссионная ошибка `e_t` лежит на стороне HSMM (`follower.hsmm.last_best_pitch_distance`), не в TempoTracker — см. ноутбук 09.

In [ ]:
print("recent_tempo_ratios:", list(tempo.recent_tempo_ratios))
print("last_variance      :", tempo.last_variance)
print("recent_control_pts :", list(tempo.recent_control_points))